In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime

# Local project utilities
import sys
sys.path.append("..")
from parcel_calculations import add_improvement_ratio_fields
from cloud_utils import get_feature_data_with_geometry, ensure_geodataframe


# Config
SCRAPE_DATA = 0  # set to 1 to rescrape from ArcGIS
DATA_DIR = "data/denver"
os.makedirs(DATA_DIR, exist_ok=True)


In [ ]:
if SCRAPE_DATA == 1:
    base_url = "https://services1.arcgis.com/zdB7qR0BtYrg0Xpl/ArcGIS/rest/services"
    dataset_name = "ODC_PROP_PARCELS_A"
    layer_id = 245  # PROP_PARCELS_A layer

    parcel_gdf = get_feature_data_with_geometry(dataset_name, base_url, layer_id)

    today_str = datetime.now().strftime("%Y_%m_%d")
    out_path = os.path.join(DATA_DIR, f"denver_parcels_{today_str}.parquet")
    parcel_gdf.to_parquet(out_path, index=False)
    print(f"✅ Saved new scrape to {out_path}")

else:
    files = glob.glob(os.path.join(DATA_DIR, "denver_parcels_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parcel files found in {DATA_DIR}. Set SCRAPE_DATA=1 to scrape.")

    files_sorted = sorted(
        files,
        key=lambda x: datetime.strptime(
            os.path.basename(x).replace("denver_parcels_", "").replace(".parquet", ""),
            "%Y_%m_%d"
        ),
        reverse=True
    )
    latest_file = files_sorted[0]
    print(f"✅ Loading most recent scrape: {latest_file}")
    parcel_gdf = pd.read_parquet(latest_file)

# Ensure GeoDataFrame
parcel_gdf = ensure_geodataframe(parcel_gdf)
print(f"✅ Loaded as {type(parcel_gdf).__name__} | CRS = {parcel_gdf.crs}")


In [ ]:
pd.set_option('display.max_columns', None)
display(parcel_gdf.head())


In [ ]:
print(parcel_gdf["PARCELNUM"].value_counts(dropna=False))


In [ ]:
for col in ["SCHEDNUM", "MAPNUM", "BLKNUM", "PARCELNUM"]:
    n_dupes = parcel_gdf.duplicated(subset=[col]).sum()
    print(f"Number of duplicate rows in '{col}': {n_dupes}")

# Create a column that concatenates MAPNUM, BLKNUM, and PARCELNUM as a string (with underscore as separator)
parcel_gdf["NUMS_CONCAT"] = (
    parcel_gdf["MAPNUM"].astype(str) + "_" +
    parcel_gdf["BLKNUM"].astype(str) + "_" +
    parcel_gdf["PARCELNUM"].astype(str)
)

# Identify duplicate rows based on NUMS_CONCAT
dupe_mask = parcel_gdf.duplicated(subset=["NUMS_CONCAT"], keep=False)
num_dupes = dupe_mask.sum()
print(f"Number of duplicate rows by NUMS_CONCAT: {num_dupes}")

# Print value counts of D_CLASS_CN among those duplicates
print("Value counts of D_CLASS_CN among duplicated NUMS_CONCAT rows:")
print(parcel_gdf.loc[dupe_mask, "D_CLASS_CN"].value_counts(dropna=False))


In [ ]:
parcel_gdf = parcel_gdf[parcel_gdf["SITUS_CITY"].str.upper() == "DENVER"]
print(f"Number of rows after filtering for DENVER: {len(parcel_gdf)}")


In [ ]:
# Print full value counts without truncation
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(parcel_gdf["D_CLASS_CN"].value_counts(dropna=False))


In [ ]:
# For Denver, collapse duplicates on NUMS_CONCAT by summing all all-caps columns (numerics),
# and taking the first string/categorical from other columns, and unioning geometries.
from shapely.ops import unary_union
from shapely.geometry import MultiPolygon
import numpy as np

if "NUMS_CONCAT" in parcel_gdf.columns:
    subset_cols = ["NUMS_CONCAT"]

    # Identify all-caps columns (as in Denver) for summing
    all_caps_cols = [col for col in parcel_gdf.columns if col.isupper() and parcel_gdf[col].dtype.kind in "biufcO"]  # O: object may still be numeric (e.g. int64 upcast)
    string_cols = [
        col for col in parcel_gdf.columns
        if not col in subset_cols + ["geometry"] and col not in all_caps_cols
    ]
    # More robust: Only actually sum if dtype is numeric
    numeric_sum_cols = [
        col for col in all_caps_cols if np.issubdtype(parcel_gdf[col].dtype, np.number)
    ]
    # Handle geometry
    def collapse_geoms(geoms):
        geoms = [g for g in geoms if g is not None]
        if not geoms:
            return None
        out = []
        while geoms:
            ref = geoms.pop(0)
            group = [ref]
            rest = []
            for g in geoms:
                if ref.intersects(g) or ref.touches(g) or ref.equals(g):
                    group.append(g)
                else:
                    rest.append(g)
            unioned = unary_union(group)
            out.append(unioned)
            geoms = rest
        if len(out) == 1:
            return out[0]
        # flatten out to MultiPolygon where possible
        polygons = []
        for g in out:
            if g.geom_type == "Polygon":
                polygons.append(g)
            elif g.geom_type == "MultiPolygon":
                polygons.extend(g.geoms)
            else:
                polygons.append(g)
        return MultiPolygon(polygons)

    # Build aggregation dictionary
    agg_dict = {}
    for col in parcel_gdf.columns:
        if col in subset_cols:
            continue
        elif col in numeric_sum_cols:
            agg_dict[col] = "sum"
        elif col == "geometry":
            agg_dict["geometry"] = collapse_geoms
        else:
            agg_dict[col] = "first"

    parcel_gdf_collapsed = parcel_gdf.groupby(subset_cols, dropna=False).agg(agg_dict).reset_index()
    print(f"Collapsed dataframe now has {len(parcel_gdf_collapsed)} rows (from {len(parcel_gdf)} original rows).")

    # Make sure GeoDataFrame type & CRS preserved
    from geopandas import GeoDataFrame
    was_geodf = isinstance(parcel_gdf, GeoDataFrame)
    crs = getattr(parcel_gdf, "crs", None) if was_geodf else None
    if "geometry" in parcel_gdf_collapsed.columns:
        parcel_gdf_collapsed = GeoDataFrame(parcel_gdf_collapsed, geometry="geometry", crs=crs)
    parcel_gdf = parcel_gdf_collapsed.copy()

    # Print how many duplicate sets were collapsed
    value_counts = parcel_gdf.groupby(subset_cols).size()
    n_multi = (value_counts > 1).sum()
    print(f"Collapsed {n_multi} sets of duplicate {subset_cols}.")

else:
    print(f"'NUMS_CONCAT' column not found in parcel_gdf.")


In [ ]:
def categorize_property_type(d_class_cn):
    """
    Assigns property category based on Denver's D_CLASS_CN value.
    """
    category_mapping = {
        # Single Family Residential
        "Single Family": [
            "SFR Grade C", "SFR Grade B", "SFR Grade A", "SFR Grade D or E", "SFR Grade X",
            "SFR Grade B w/RK", "SFR Grade C, D, or E, w/RK", "SFR Grade A or X, w/RK"
        ],
        # Condo and townhouse/rowhouse
        "Condo/Townhouse/Rowhouse": [
            "RESIDENTIAL-CONDOMINIUM", "RESIDENTIAL CONDOMINIUM", "COMMERCIAL-CONDOMINIUM",
            "RESIDENTIAL-ROWHOUSE"
        ],
        # Duplex/triplex/quad (small multi-family)
        "Small Multi-Family (2-4 units)": [
            "RESIDENTIAL-DUPLEX", "RESIDENTIAL-TRIPLEX", "RESIDENTIAL-4 TO 8 UNITS"
        ],
        # Large Multi-Family (>4 units, apts)
        "Large Multi-Family (5+ units)": [
            "RESIDENTIAL-APARTMENT", "RESIDENTIAL-MULTI UNIT APTS", "RESIDENTIAL-SENIOR CITIZEN APT"
        ],
        # General catch-all for Residential
        "Other Residential": [
            "RESIDENTIAL", "RESIDENTIAL LAND CONTIGUOUS", "RESIDENTIAL  LAND FOR LAND/ IMPS PARCEL",
            "RESIDENTIAL-MISC IMPS", "RESIDENTIAL GRACE YEAR", "RESIDENTIAL-BOARDING HOME",
            "RESIDENTIAL-NURSING FACILITY"
        ],
        # Vacant/land
        "Vacant Land": [
            "VACANT LAND", "VACANT LAND /GENERAL COMMON ELEMENTS", "VACANT LAND W/MINOR STRUCTURE",
            "GENERAL COMMON ELEMENTS", "RESIDENTIAL LAND CONTIGUOUS"
        ],
        # Manufactured housing, mobile homes, MH land, park
        "Mobile Home": [
            "MH / Minor Structures", "MOBILE HOME LAND", "Mobile Home Park"
        ],
        # Agricultural/Rural/Open
        "Agricultural/Open": [
            "DRY FARM LAND", "CUR - USE - AG", "Agricultural", "Agricultural Not Classified", "GOLF COURSE"
        ],
        # Commercial
        "Commercial": [
            "COMMERCIAL-RETAIL", "COMMERCIAL-OFFICE", "COMMERCIAL-MISC IMPS", "COMMERCIAL-RESTAURANT",
            "COMMERCIAL-SHOPPING CENTER", "COMMERCIAL-MEDICAL OFFICE", "COMMERCIAL-CONDOMINIUM",
            "COMMERCIAL-HOTEL", "COMMERCIAL-PARKING GARAGE", "COMMERCIAL-MOTEL", "COMMERCIAL",
            "COMMERCIAL-THEATER", "COMMERCIAL-FINANCIAL OFFICE", "RETAIL W/MIXED USE",
            "COMMERCIAL-MISC IMPS", "COMMERCIAL-RETAIL", "COMMERCIAL-OFFICE", "RESTAURANT W/MIXED USE",
            "HOTEL W/MIXED USE", "WAREHOUSE W/MIXED USE", "OFFICE W/MIXED USE", "MOTEL W/MIXED USE",
            "SHOPPING CENTER W/MIXED USE", "FINANCIAL OFFICE W/MIXED USE", "OTHER COMMERCIAL P.I.",
            "COMMERCIAL-CONDOMINIUM"
        ],
        # Industrial/manufacturing/warehouse/auto
        "Industrial/Manufacturing": [
            "INDUSTRIAL-WAREHOUSE", "INDUSTRIAL-AUTO SERVICE GARAGE", "INDUSTRIAL-CHURCH",
            "INDUSTRIAL-SCHOOL", "INDUSTRIAL-AUTO DEALER", "INDUSTRIAL-FACTORY", "INDUSTRIAL-CONV STORE W/PUMPS",
            "INDUSTRIAL-MISC RECREATION", "INDUSTRIAL-SERVICE STATION", "INDUSTRIAL-PRESCHOOL",
            "INDUSTRIAL-MEETING HALL", "INDUSTRIAL-CAR WASH", "INDUSTRIAL-VETERINARY",
            "INDUSTRIAL-FOOD PROCESSING", "INDUSTRIAL-CONVERTED CHURCH", "INDUSTRIAL-CEMETERY BLDG",
            "INDUSTRIAL-SHIPPING TERMINAL", "INDUSTRIAL-MORTUARY", "INDUSTRIAL-HEALTH CLUB",
            "INDUSTRIAL-PRINTING PLANT", "INDUSTRIAL-MEAT PACKING", "INDUSTRIAL-DRY CLEANING",
            "INDUSTRIAL-GREENHOUSE", "INDUSTRIAL-GRAIN ELEVATOR", "INDUSTRIAL-CITY CLUB", "INDUSTRIAL-BOWLING ALLEY",
            "INDUSTRIAL-BRICK PLANT", "FACTORY W/MIXED USE"
        ],
        # Special Purpose/Governmental/Education/Institutional
        "Civic/Institutional": [
            "SPECIAL PURPOSE", "FIRE STATION", "POLICE-FIRE STATION", "COUNTY JAIL", "STOCK SHOW",
            "DENVER PARK", "STADIUM", "STADIUM-MIXED USE", "AIRPORT P.I. RETAIL", "AIRPORT P.I. OTHER",
            "SCHOOL", "SOCIAL/RECREATION W/MIXED USE", "MEDICAL OFFICE W/MIXED USE", "ENTERTAINMENT P.I.",
            "PUBLIC ASSEMBLY", "OTHER CULTURAL", "PARK", "RECREATION P.I."
        ],
        # Utilities/Transportation
        "Transportation/Utilities": [
            "TRANSPORTATION", "UTILITIES", "COMMUNICATION"
        ],
        # Mixed Use (if explicitly stated)
        "Mixed Use": [
            "RETAIL W/MIXED USE", "OFFICE W/MIXED USE", "RESTAURANT W/MIXED USE", "HOTEL W/MIXED USE", 
            "WAREHOUSE W/MIXED USE", "MOTEL W/MIXED USE", "SHOPPING CENTER W/MIXED USE", 
            "FINANCIAL OFFICE W/MIXED USE", "MISC IMPROVEMENTS W/MIXED USE", 
            "FOOD PROCESSING W/MIXED USE", "VETERINARY W/MIXED USE", "THEATER W/MIXED USE"
        ],
        # Possessory interests
        "Possessory Interest": [
            "POSSESSORY INTEREST"
        ],
        # Catch-all for any kind of "None" or missing
        "Unknown": [
            None, "None"
        ]
    }

    for category, descriptions in category_mapping.items():
        if d_class_cn in descriptions:
            return category

    # Heuristic mapping for common substring occurrences
    s = str(d_class_cn).upper() if d_class_cn is not None else ""
    if "VACANT" in s or "LAND" in s:
        return "Vacant Land"
    if "RESIDENTIAL" in s:
        if "CONDOMINIUM" in s or "ROWHOUSE" in s:
            return "Condo/Townhouse/Rowhouse"
        if "DUPLEX" in s or "TRIPLEX" in s or "4 TO 8" in s:
            return "Small Multi-Family (2-4 units)"
        if "APARTMENT" in s or "MULTI UNIT" in s or "SENIOR CITIZEN" in s:
            return "Large Multi-Family (5+ units)"
        return "Other Residential"
    if "COMMERCIAL" in s:
        return "Commercial"
    if "INDUSTRIAL" in s:
        return "Industrial/Manufacturing"
    if "MOBILE HOME" in s or "MH" in s:
        return "Mobile Home"
    if "AGRICULTURAL" in s or "FARM" in s:
        return "Agricultural/Open"
    if "PARK" in s or "GOLF" in s:
        return "Civic/Institutional"
    if "SCHOOL" in s or "JAIL" in s or "HOSPITAL" in s or "POLICE" in s or "FIRE" in s:
        return "Civic/Institutional"
    if "MIXED USE" in s:
        return "Mixed Use"

    return "Other"

# Apply the function to the DataFrame
parcel_gdf['PROPERTY_CATEGORY'] = parcel_gdf['D_CLASS_CN'].apply(categorize_property_type)

In [ ]:
# No jurisdictional filtering needed for Denver; keep all parcels
print(f"ℹ️ Retaining all Denver parcels: {len(parcel_gdf):,} records")


In [ ]:
# -----------------------------
# 1) Clone dataframe
# -----------------------------
export_gdf = parcel_gdf.copy()

# -----------------------------
# 2) Exclude fully exempt parcels and flag
# -----------------------------
if "full_exmp" not in export_gdf.columns:
    if "EXEMPT_AMT_LOCAL" in export_gdf.columns and "ASSESSED_TOTAL_VALUE_LOCAL" in export_gdf.columns:
        assessed_total = export_gdf["ASSESSED_TOTAL_VALUE_LOCAL"].fillna(0)
        exempt_amt = export_gdf["EXEMPT_AMT_LOCAL"].fillna(0)
        ratio = exempt_amt / assessed_total.replace(0, np.nan)
        full_exmp_bool = (ratio >= 0.995) | ((assessed_total <= 0) & (exempt_amt > 0))
        export_gdf["full_exmp"] = full_exmp_bool.astype(int)
    elif "TAXABLE_AMT_LOCAL" in export_gdf.columns:
        export_gdf["full_exmp"] = (export_gdf["TAXABLE_AMT_LOCAL"] <= 0).astype(int)
    else:
        raise ValueError("Need taxable/exempt fields to define exemptions.")

# Drop fully exempt parcels
export_gdf = export_gdf[export_gdf["full_exmp"] == 0].copy()
export_gdf["exemption_flag"] = (export_gdf["full_exmp"] == 1).astype(int)

# -----------------------------
# 3) Use appraised land/improvement values (fallback to assessed)
# -----------------------------
if "APPRAISED_LAND_VALUE" in export_gdf.columns:
    export_gdf["land_value"] = export_gdf["APPRAISED_LAND_VALUE"]
elif "ASSESSED_LAND_VALUE_LOCAL" in export_gdf.columns:
    export_gdf["land_value"] = export_gdf["ASSESSED_LAND_VALUE_LOCAL"]
else:
    raise ValueError("Need land value field for Denver parcels.")

if "APPRAISED_IMP_VALUE" in export_gdf.columns:
    export_gdf["improvement_value"] = export_gdf["APPRAISED_IMP_VALUE"]
elif "ASSESSED_BLDG_VALUE_LOCAL" in export_gdf.columns:
    export_gdf["improvement_value"] = export_gdf["ASSESSED_BLDG_VALUE_LOCAL"]
else:
    export_gdf["improvement_value"] = 0

# -----------------------------
# 4) Ensure PROPERTY_CATEGORY exists
# -----------------------------
if "PROPERTY_CATEGORY" not in export_gdf.columns:
    if "D_CLASS_CN" in export_gdf.columns:
        export_gdf["PROPERTY_CATEGORY"] = np.where(
            export_gdf["D_CLASS_CN"].str.contains("Vacant", case=False, na=False),
            "Vacant Land",
            "Other"
        )
    elif "PROP_CLASS" in export_gdf.columns:
        export_gdf["PROPERTY_CATEGORY"] = np.where(
            export_gdf["PROP_CLASS"].astype(str).str.contains("VAC", case=False, na=False),
            "Vacant Land",
            "Other"
        )
    else:
        export_gdf["PROPERTY_CATEGORY"] = "Other"

export_gdf["property_land_use_category"] = export_gdf["PROPERTY_CATEGORY"]

# -----------------------------
# 5) Refined land use classification
# -----------------------------
def categorize_property_refined(row):
    cat = str(row["PROPERTY_CATEGORY"])
    if "Vacant" in cat:
        return "Vacant"
    elif "Parking" in cat:
        return "Parking Lot"
    elif row["improvement_value"] < 0.5 * (row["land_value"] + row["improvement_value"]):
        return "Underdeveloped"
    else:
        return None

export_gdf["property_land_use_refined"] = export_gdf.apply(categorize_property_refined, axis=1)

# -----------------------------
# 6) Compute parcel area sqft
# -----------------------------
if "Shape__Area" in export_gdf.columns:
    export_gdf["area_sqft"] = export_gdf["Shape__Area"]
else:
    export_gdf["area_sqft"] = export_gdf.geometry.area

export_gdf["area_sqft"] = export_gdf["area_sqft"].replace(0, np.nan)

# -----------------------------
# 7) Per sqft metrics and full market value per sqft
# -----------------------------
if "APPRAISED_TOTAL_VALUE" in export_gdf.columns:
    export_gdf["full_market_value"] = export_gdf["APPRAISED_TOTAL_VALUE"]
elif "ASSESSED_TOTAL_VALUE_LOCAL" in export_gdf.columns:
    export_gdf["full_market_value"] = export_gdf["ASSESSED_TOTAL_VALUE_LOCAL"]
else:
    export_gdf["full_market_value"] = export_gdf.get("land_value", 0) + export_gdf.get("improvement_value", 0)

export_gdf["full_market_value_per_sqft"] = export_gdf["full_market_value"] / export_gdf["area_sqft"]
export_gdf["land_value_per_sqft"] = export_gdf["land_value"] / export_gdf["area_sqft"]
export_gdf["improvement_value_per_sqft"] = export_gdf["improvement_value"] / export_gdf["area_sqft"]

# -----------------------------
# 8) Derived improvement/land ratios
# -----------------------------
export_gdf = add_improvement_ratio_fields(
    export_gdf,
    land_col="land_value",
    improvement_col="improvement_value"
)

# -----------------------------
# Save the link, ensuring it's present in the export data
# -----------------------------
if "link" not in export_gdf.columns:
    export_gdf["link"] = np.nan

# -----------------------------
# 9) Select columns (no current_tax), now including link
# -----------------------------
columns_to_export = [
    "geometry",
    "exemption_flag",
    "property_land_use_category",
    "property_land_use_refined",
    "full_market_value",
    "full_market_value_per_sqft",
    "land_value",
    "land_value_per_sqft",
    "improvement_value",
    "improvement_value_per_sqft",
    "TLLDIMPROV",
    "IMPR_LAND_RATIO",
    "IMPR_LAND_PCT",
    "IMPR_PCT_TOTAL",
    "link"
]

# Guarantee the export schema even if some columns are absent
for col in columns_to_export:
    if col not in export_gdf.columns:
        export_gdf[col] = np.nan

export_final = export_gdf[columns_to_export].rename(columns={
    "land_value": "current_full_land_value"
})

# Ensure geometry validity
export_final["geometry"] = export_final["geometry"].apply(
    lambda geom: geom if geom is None or geom.is_valid else geom.buffer(0)
)

# Ensure CRS is EPSG:4326
export_final = gpd.GeoDataFrame(export_final, geometry="geometry", crs=export_gdf.crs)
if export_final.crs is None or export_final.crs.to_epsg() != 4326:
    export_final = export_final.to_crs("EPSG:4326")
    print("✅ Converted to EPSG:4326")

# -----------------------------
# 10) Save Parquet: both canonical and dated version
# -----------------------------
canonical_path = os.path.join(DATA_DIR, "denver-co-parcels.parquet")
today_str = datetime.now().strftime("%Y_%m_%d")
dated_path = os.path.join(DATA_DIR, f"denver-co-parcels_{today_str}.parquet")

export_final.to_parquet(canonical_path, index=False)
export_final.to_parquet(dated_path, index=False)

print(f"✅ Saved export parquet: {canonical_path}")
print(f"✅ Also saved dated version: {dated_path}")
print("Export columns:", export_final.columns.tolist())
print("\nRefined category counts:")
print(export_final["property_land_use_refined"].value_counts(dropna=False))


In [ ]:
# Optional: upload export_final to dev Azure blob
upload_dev = True

if upload_dev:
    from azure.storage.blob import BlobServiceClient

    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError(
            "Set AZURE_STORAGE_CONNECTION_STRING or update connection_string before upload."
        )

    container = os.getenv("AZURE_DEV_CONTAINER", "parquets-dev")
    blob_name = "denver-co-parcels.parquet"
    local_path = os.path.join(DATA_DIR, blob_name)

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Local parquet not found: {local_path}")

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    container_client = blob_service.get_container_client(container)

    with open(local_path, "rb") as handle:
        container_client.upload_blob(name=blob_name, data=handle, overwrite=True)

    print(f"✅ Uploaded {local_path} -> {container}/{blob_name}")
else:
    print("upload_dev is False; skipping upload.")


In [36]:
upload_dev_pmtiles = True  # Only run PMTiles conversion/upload if this is True

if upload_dev_pmtiles:
    # Convert parquet to PMTiles and upload to Azure
    import subprocess
    import sys
    from pathlib import Path

    # Get the path to the conversion script
    # Notebook is in data/jurisidictions/, script is in data/scripts/
    # Script expects to be run from PROJECT ROOT (where data/ is a subdirectory)
    notebook_dir = Path.cwd()  # Current working directory when notebook runs
    
    # Find project root by looking for data/scripts/ directory
    current = notebook_dir
    project_root = None
    while current.parent != current:
        if (current / "data" / "scripts" / "parquet_to_pmtiles.py").exists():
            project_root = current
            break
        current = current.parent
    
    if not project_root:
        # Fallback: assume project root is 2 levels up from jurisidictions
        project_root = notebook_dir.parent.parent if notebook_dir.name == "jurisidictions" else notebook_dir.parent
    
    script_path = project_root / "data" / "scripts" / "parquet_to_pmtiles.py"
    
    if not script_path.exists():
        raise FileNotFoundError(f"Could not find parquet_to_pmtiles.py script at: {script_path}")

    # Verify parquet file exists at expected location (relative to project root)
    expected_parquet = project_root / "data" / "jurisidictions" / "data" / "denver" / "denver-co-parcels.parquet"
    print(f"Running PMTiles conversion script: {script_path}")
    print(f"Project root (working directory): {project_root}")
    print(f"Expected parquet: {expected_parquet}")
    print(f"Parquet exists: {expected_parquet.exists()}")

    # Run the conversion script
    cmd = [
        sys.executable,
        str(script_path),
        "--city", "denver",
        "--upload",
        "--overwrite"
    ]

    # Check if connection string is set
    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        print("⚠️  AZURE_STORAGE_CONNECTION_STRING not set. Skipping upload.")
        cmd.remove("--upload")

    # Run from PROJECT ROOT (script looks for data/jurisidictions/data/denver/denver-co-parcels.parquet relative to project root)
    result = subprocess.run(cmd, cwd=str(project_root), capture_output=True, text=True)

    if result.returncode == 0:
        print("✅ PMTiles conversion and upload completed successfully!")
        print("✅ PMTiles file: denver-co-parcels.pmtiles")
        print("✅ Metadata file: denver-co-parcels-metadata.json")
        if result.stdout:
            print("\nScript output:")
            print(result.stdout)
    else:
        print(f"❌ PMTiles conversion failed with exit code {result.returncode}")
        if result.stderr:
            print(f"\nError output:\n{result.stderr}")
        if result.stdout:
            print(f"\nStandard output:\n{result.stdout}")
else:
    print("upload_dev_pmtiles is False; skipping PMTiles conversion and upload.")


Running PMTiles conversion script: ./data/scripts/parquet_to_pmtiles.py
Project root (working directory): .
Expected parquet: ./data/jurisidictions/data/denver/denver-co-parcels.parquet
Parquet exists: True
✅ PMTiles conversion and upload completed successfully!
✅ PMTiles file: denver-co-parcels.pmtiles
✅ Metadata file: denver-co-parcels-metadata.json

Script output:
Step 1: Loading parquet and computing metadata
Loaded 227,609 features
Computed metadata: 13 fields, 2 refined categories
✅ Metadata saved: data/jurisidictions/data/denver/denver-co-parcels-metadata.json
Step 2: Checking dependencies
✅ All dependencies found
Step 3: Converting to PMTiles
Loading parquet: data/jurisidictions/data/denver/denver-co-parcels.parquet
Loaded 227,609 features
Writing GeoJSON: /var/folders/jb/s1bhbc2x3dgbhdvtbnz335sc0000gn/T/tmp1dvip7tg/input.geojson
✅ GeoJSON written: /var/folders/jb/s1bhbc2x3dgbhdvtbnz335sc0000gn/T/tmp1dvip7tg/input.geojson
Creating MBTiles: /var/folders/jb/s1bhbc2x3dgbhdvtbnz335